# Live: OFDM with a Schmidl-and-Cox synchroniser, in the P501 FPGA, against its own DAC or a B210

The receiver of the overlay **`p501_ofdm_rx_sc`** decodes a continuous stream of OFDM frames, 3.84 MS/s, **all in the FPGA**: Schmidl-and-Cox detector (delay 32, plateau rule) -> coarse carrier (CORDIC) and gain -> derotation ->
64-point FFT -> channel estimate from two QPSK training symbols -> per-subcarrier equaliser -> pilot gain and slope tracking -> 16-QAM slicer -> PRBS-15 symbol and bit error counters.
The numbers are bit-exact with the integer model `tutorial/ofdm/ofdm_fixed.py` (design and results: `tutorial/ofdm/PLAN_OFDM.md`).

**The frame** (3840 samples = 1.0015 ms, about 1000 frames per second, 4224 bits = 4.2 Mb/s): `S&C symbol (80) | T1 (80) | T2 (80) | 44 data symbols of 80 samples (16 cyclic prefix + 64) | 80 idle`.
N = 64 at 3.84 MS/s: 60 kHz spacing, 28 used subcarriers (k = -14 .. -1, +1 .. +14), 4 QPSK **pilots** at k = -11, -4, +4, +11 and 24 **16-QAM data** subcarriers.
The S&C symbol (even subcarriers only, periodic with 32 samples) gives the frame start and the carrier offset (+-60 kHz; the natural offset of the P501 and the B210 is **+9 kHz**, 0.15 of the spacing);
T1 and T2 give the channel; the pilots follow the common phase error and the gain from symbol to symbol.

**The transmitter** here is the board's DAC: the notebook sends the 3840-sample frame as a *cyclic buffer* through the DAC DMA (`ofdm_frame.npy`, made by `tutorial/ofdm/make_ofdm_frame.py`).
**Three ways to use it:** (1) **digital loopback** inside the AD9361: nothing is radiated; (2) **TX1 -> RX1 over the air** (antennas a few centimetres apart or a cable with an attenuator): TX atten 20 dB and more; (3) the **B210** as the transmitter
(`python3 tutorial/ofdm/ofdm_frame_tx.py --freq 850e6 --tx-gain 79 --rms 0.25 --seconds 600` on the host) and the P501 as the receiver.

**What the picture shows:** the **constellation** of the first 1024 corrected data symbols of a frame (red crosses: the 16 ideal points); the **status** (frames per second, symbol and bit error rates, EVM, SNR, carrier offset, gain, lateness);
the **channel |H| (dB)** of the 28 subcarriers, the **EVM per data subcarrier**, and strips with the carrier offset, the frames per second and log10 of the symbol error rate.

## Configuration, overlay and bring-up

In [ ]:
BIT       = "/home/xilinx/jupyter_notebooks/p501_ofdm/p501_ofdm_rx_sc.bit"      # OFDM receiver (the .hwh next to it)
FRAME_NPY = "/home/xilinx/jupyter_notebooks/p501_ofdm/ofdm_frame.npy"           # the 3840-sample frame (unit power subcarriers)
FS        = 3_840_000          # sample rate
LO        = 850_000_000        # carrier, Hz: the same as the B210's --freq (915 MHz gave a weak B210 signal in our lab, 850 MHz a strong one); change it and run this cell and the next again
RX_GAIN   = 70                 # P501 RX gain-table index 0..76
TX_ATTEN0 = 89.75              # TX attenuation (dB) at the start and at the end: the transmitter is silent

In [ ]:
import sys, time, os, io, threading, traceback
import numpy as np
from pynq import Overlay, allocate

sys.path.insert(0, '/home/xilinx/jupyter_notebooks')
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/p501_fft_dds')            # p501_radio.py, axi_dmac.py
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/p501_ofdm')               # p501_ofdm_rx.py
from axi_dmac import AxiDmac
import p501_radio
import importlib, p501_ofdm_rx
importlib.reload(p501_ofdm_rx)                                               # a module that is already imported is not re-read: reload, so that a newer file is used without restarting the kernel
from p501_ofdm_rx import P501OfdmRx, USED, R_UNIT
print('driver:', p501_ofdm_rx.__file__)
_need = [m for m in ("configure", "start", "stop", "arm_capture", "poll_capture", "wait_frames", "stats", "capture", "channel") if not hasattr(P501OfdmRx, m)]
assert not _need, f"p501_ofdm_rx.py is older than this notebook (missing {_need}): copy notebook/p501_ofdm_rx.py again to /home/xilinx/jupyter_notebooks/p501_ofdm/ and RESTART THE KERNEL"

results = {}
def check(name, ok, detail):
    results[name] = (bool(ok), detail)
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {detail}")

ol = Overlay(BIT)                                                                  # programs the PL
rx = P501OfdmRx(ol.ofdm_rx)                                                        # receiver registers at 0x9C470000 (the constructor checks the ID 0xDD520007)
txd = AxiDmac(ol.axi_ad9361_dac_dma)                                               # the base design's DAC DMA: the transmitter
ad = p501_radio.bring_up(ol, lo_hz=LO, tx_atten_db=TX_ATTEN0, rx_gain_index=RX_GAIN)   # the AD9361 over SPI; TX attenuation at its maximum during the calibrations
rx.stop(); txd.stop()
print("bring-up ok:", ad.info["ok"], "| receiver ID 0x%08X" % rx.ip.read(0), "| LO %.3f MHz, RX gain index %d" % (LO / 1e6, RX_GAIN))

frame = np.load(FRAME_NPY).astype(np.complex64)                                    # 3840 samples, subcarriers of unit power: rms sqrt(28/64) = 0.66
_tx = {"buf": None}
def tx_start(level):
    """send the frame as a cyclic DMA buffer; level = the rms at the ADC in counts for the digital loopback (the DAC takes 12 bits left-justified: about 16 x level in the buffer); over the air it is the transmit power"""
    tx_stop()
    k = level * 16.0 / np.sqrt(np.mean(np.abs(frame) ** 2))
    buf = allocate(shape=(len(frame) * 2,), dtype=np.int16)
    buf[0::2] = np.clip(np.round(frame.real * k * np.sqrt(28 / 64)), -32767, 32767).astype(np.int16)
    buf[1::2] = np.clip(np.round(frame.imag * k * np.sqrt(28 / 64)), -32767, 32767).astype(np.int16)
    buf.flush(); txd.transfer_from_memory(buf.physical_address, buf.nbytes, cyclic=True); _tx["buf"] = buf
def tx_stop():
    txd.stop()
    if _tx["buf"] is not None: _tx["buf"].freebuffer(); _tx["buf"] = None

## Self-test: the DAC into the receiver, inside the AD9361 (digital loopback)

No antennas, no B210, **nothing radiated** (the TX attenuation stays at its maximum). First the **transmitter is off**: the receiver must not find a frame in the noise. Then the DAC sends the frame at three levels
(65, 194 and 389 counts rms at the ADC): the receiver must find **every frame** (about 2003 in 2 s), reject none, and decode the 44 x 24 = 1056 data symbols of every frame **without an error**, with an EVM better than -35 dB and nothing late.

In [ ]:
def measure(seconds=2.0, settle=0.3):
    """differences of the cumulative counters over `seconds`, after the receiver has found its first frames"""
    ok = rx.wait_frames(3, timeout_s=2.0); time.sleep(settle); a = rx.stats(); time.sleep(seconds); b = rx.stats()
    d = lambda k: b[k] - a[k]
    return ok, a, b, d

p501_radio.loopback(ad, True); ad.set_tx_atten_db(TX_ATTEN0)
tx_stop(); rx.stop(); rx.configure(); rx.start(continuous=True, capture=True); time.sleep(0.5); s0 = rx.stats(); rx.stop()
check("no transmitter: no frame, no false alarm", s0["frames"] == 0 and s0["found"] == 0, f"frames {s0['frames']}, found {s0['found']}, rejected {s0['rejected']}")

for lvl in (65, 194, 389):
    rx.stop(); tx_start(lvl); time.sleep(0.05); rx.configure(); rx.start(continuous=True, capture=True)       # stop the receiver while the DMA buffer changes: the cut-over frame would set the sticky overflow flag
    ok, a, b, d = measure(2.0)
    check(f"loopback {lvl} counts: every frame found", ok and abs(d("frames") - 2003) <= 6 and d("rejected") == 0 and d("bad_frames") == 0, f"{d('frames')} frames in 2 s (about 2003), found {d('found')}, rejected {d('rejected')}, bad {d('bad_frames')}")
    check(f"loopback {lvl} counts: payload without an error", d("tot_err") == 0 and d("tot_sym") > 2_000_000, f"{d('tot_err')} symbol errors in {d('tot_sym'):,} symbols, {d('tot_bit')} bit errors")
    check(f"loopback {lvl} counts: EVM", b["evm_db"] < -35, f"{b['evm_db']:.1f} dB, gain {b['gain']:.2f} = level {b['level_counts']:.0f} counts, carrier {b['carrier_hz']:+.0f} Hz")
    check(f"loopback {lvl} counts: nothing late", b["late_count"] == 0 and not (b["ovf"] or b["fft_ovf"] or b["overrun"]), f"late {b['late_count']}, ovf {b['ovf']}, fft_ovf {b['fft_ovf']}, overrun {b['overrun']}")
z = rx.capture(); k_, h_ = rx.channel()
check("capture RAM and channel estimate", len(z) == 1024 and abs(np.sqrt(np.mean(np.abs(z) ** 2)) - 1.0) < 0.05 and len(h_) == 28, f"{len(z)} soft symbols, rms {np.sqrt(np.mean(np.abs(z) ** 2)):.3f} (unit), |H| {np.abs(h_).min():.0f} .. {np.abs(h_).max():.0f}")
rx.stop(); tx_stop(); p501_radio.loopback(ad, False)

## The live machinery

* `render()` draws one picture with NumPy and Pillow: the constellation, the status, the channel `|H|`, the EVM per data subcarrier and three strips.
* `Live` runs `read registers -> read capture RAM and channel estimate -> picture` in a background thread, so the controls stay responsive.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image as IPImage
from PIL import Image as PILImage, ImageDraw, ImageFont

W, H = 900, 700
CX, CY, SC = 230, 215, 125                          # constellation centre and pixels per unit (axes +-1.6)
FONT = ImageFont.load_default()
HIST = 80
LEV = np.array([-3, -1, 1, 3]) / np.sqrt(10)
PILOTS = (-11, -4, 4, 11)
DATA_K = [int(k) for k in USED if int(k) not in PILOTS]
hist_off = [np.nan] * HIST; hist_fr = [np.nan] * HIST; hist_ser = [np.nan] * HIST

def base_frame():
    img = PILImage.new("RGB", (W, H), (255, 255, 255)); d = ImageDraw.Draw(img)
    d.rectangle([CX - 1.6 * SC, CY - 1.6 * SC, CX + 1.6 * SC, CY + 1.6 * SC], outline=(90, 90, 90))
    for u in (-1, 0, 1):
        d.line([CX + u * SC, CY - 1.6 * SC, CX + u * SC, CY + 1.6 * SC], fill=(225, 225, 225)); d.line([CX - 1.6 * SC, CY - u * SC, CX + 1.6 * SC, CY - u * SC], fill=(225, 225, 225))
    d.text((CX - 110, CY + 1.6 * SC + 6), "corrected data symbols (first 1024 of a frame)", fill=(60, 60, 60), font=FONT)
    return img
BASE = base_frame()

def strip(d, x0, y0, w, h, series, label, lo=None, hi=None, fmt="{:.0f}"):
    d.rectangle([x0, y0, x0 + w, y0 + h], outline=(150, 150, 150)); d.text((x0, y0 - 12), label, fill=(60, 60, 60), font=FONT)
    allv = np.concatenate([np.asarray(s, dtype=float) for s, _ in series]); allv = allv[~np.isnan(allv)]
    if lo is None:
        c = float(np.median(allv)) if len(allv) else 0.0; span = max(2000.0, 1.2 * (float(allv.max() - allv.min()) if len(allv) else 0.0))
        lo, hi = c - span / 2, c + span / 2
    for data, color in series:
        pts = [(x0 + i * w / (len(data) - 1), y0 + h - (min(max(v, lo), hi) - lo) / (hi - lo) * h) for i, v in enumerate(data) if not np.isnan(v)]
        if len(pts) > 1: d.line(pts, fill=color, width=1)
    d.text((x0 + w + 4, y0 - 2), fmt.format(hi), fill=(100, 100, 100), font=FONT); d.text((x0 + w + 4, y0 + h - 9), fmt.format(lo), fill=(100, 100, 100), font=FONT)

def nearest_err2(z):
    z = np.asarray(z)
    return np.min(np.abs(z.real[..., None] - LEV), axis=-1) ** 2 + np.min(np.abs(z.imag[..., None] - LEV), axis=-1) ** 2

def soft_evm_db(z):
    """EVM of the soft symbols against the nearest ideal point"""
    return 10 * np.log10(max(float(np.mean(nearest_err2(z))), 1e-12))

def render(z, hh, st, frate, ser_recent, ser_total, note=""):
    img = BASE.copy(); d = ImageDraw.Draw(img)
    for a, b in zip(np.asarray(z).real, np.asarray(z).imag):
        x, yy = CX + a * SC, CY - b * SC
        if abs(a) < 1.6 and abs(b) < 1.6: d.rectangle([x - 1, yy - 1, x + 1, yy + 1], fill=(31, 119, 180))
    for a in LEV:                                                                   # the 16 ideal points, on top of the symbols
        for b in LEV:
            x, y = CX + a * SC, CY - b * SC
            d.line([x - 7, y, x + 7, y], fill=(220, 40, 40)); d.line([x, y - 7, x, y + 7], fill=(220, 40, 40))
    X = 490
    ok = st["running"] and frate > 500
    d.text((X, 20), "DECODING: frames found" if ok else ("receiver running, no frames" if st["running"] else "receiver stopped"), fill=(30, 140, 50) if ok else (200, 40, 40), font=FONT)
    d.text((X, 36), note, fill=(110, 110, 110), font=FONT)
    fmt = lambda v: "-" if v != v else (f"{v:.1e}" if v else "0")
    lines = [f"frames {st['frames']:>10,}  ({frate:.0f}/s of about 1000)",
             f"found {st['found']:,}  rejected {st['rejected']:,}  bad frames {st['bad_frames']:,}",
             f"symbols checked {st['tot_sym']:>12,}   errors {st['tot_err']:>9,}",
             f"SER {fmt(ser_total)} since start,  {fmt(ser_recent)} last 2 s",
             f"EVM {st['evm_db']:.1f} dB (last frame)  {soft_evm_db(z):.1f} dB (picture)",
             f"SNR {st['snr_db']:.1f} dB (from the channel estimate and the noise sum)",
             f"carrier {st['carrier_hz']:+8.0f} Hz  (coarse {st['carrier_coarse_hz']:+.0f} Hz)",
             f"level {st['level_counts']:.0f} counts rms at the ADC   gain {st['gain']:.2f}",
             f"late {st['late_count']}   ovf {int(st['ovf'])}  fft_ovf {int(st['fft_ovf'])}  overrun {int(st['overrun'])}"]
    for i, l in enumerate(lines): d.text((X, 62 + 16 * i), l, fill=(0, 0, 0), font=FONT)
    strip(d, X, 240, 340, 80, [(hist_off, (30, 150, 60))], "carrier offset (Hz)")
    strip(d, X, 360, 340, 80, [(hist_fr, (30, 150, 60))], "frames per second (0 .. 1200)", 0, 1200)
    strip(d, X, 480, 340, 80, [(hist_ser, (180, 40, 40))], "log10 symbol error rate, last 0.5 s (-7 .. 0)", -7, 0)
    # the channel |H| (dB, relative to the mean) of the 28 used subcarriers, and the EVM per data subcarrier
    y0 = 500; x0 = 20; w = 420
    if hh is not None and len(hh) == 28 and np.all(np.abs(hh) > 0):
        mag = 20 * np.log10(np.abs(hh)); mag -= mag.mean()
        d.rectangle([x0, y0, x0 + w, y0 + 70], outline=(150, 150, 150)); d.text((x0, y0 - 12), "channel |H| of the 28 subcarriers (dB, relative to the mean, -15 .. +15)", fill=(60, 60, 60), font=FONT)
        pts = [(x0 + (i + 0.5) * w / 28, y0 + 70 - (min(max(m, -15), 15) + 15) / 30 * 70) for i, m in enumerate(mag)]
        d.line(pts, fill=(31, 119, 180), width=2)
        for i, k in enumerate(USED):
            if int(k) in PILOTS: d.line([x0 + (i + 0.5) * w / 28, y0 + 66, x0 + (i + 0.5) * w / 28, y0 + 70], fill=(220, 40, 40), width=2)
    y1 = 610
    zs = np.asarray(z)[:1008].reshape(42, 24); e = 10 * np.log10(np.maximum(nearest_err2(zs).mean(axis=0), 1e-6))      # -60 .. -5 dB
    d.rectangle([x0, y1, x0 + w, y1 + 70], outline=(150, 150, 150)); d.text((x0, y1 - 12), "EVM per data subcarrier (dB, -45 .. -5; bars up = worse)", fill=(60, 60, 60), font=FONT)
    for i, v in enumerate(e):
        hbar = (min(max(v, -45), -5) + 45) / 40 * 70; bx = x0 + (i + 0.15) * w / 24
        d.rectangle([bx, y1 + 70 - hbar, bx + 0.7 * w / 24, y1 + 70], fill=(180, 40, 40))
    buf = io.BytesIO(); img.save(buf, format="PNG", compress_level=1)
    return buf.getvalue()

class Live:
    """Background thread: registers + capture RAM -> picture."""
    def __init__(self, image, status):
        self.image, self.status = image, status
        self.running, self.thread, self.frames = False, None, 0
        self.last, self.last_t = None, time.time()
        self.st = None; self.base = None
        self.z = np.zeros(1024, dtype=complex); self.hh = None; self.t_cap = time.time()
        self.win = []                                            # (time, symbols, errors) of the last 2 s
        self.paused = False                                      # set while a start procedure or an experiment uses the registers
    def _run(self):
        t_rate = time.time(); n_rate = 0; rate = 0.0; t_h = 0.0
        while self.running:
            if self.paused:
                time.sleep(0.05); continue
            try:
                st = rx.stats()
                if st["running"]:
                    got = rx.poll_capture()                              # a new frame, or None; re-arms the capture when a rejected frame has eaten the previous arm
                    if got is not None: self.z, self.hh = got; self.t_cap = time.time()
                now = time.time(); dt = max(now - self.last_t, 1e-3)
                if self.last is not None and st["frames"] >= self.last[0]:
                    frate = (st["frames"] - self.last[0]) / dt
                    dsym, derr = st["tot_sym"] - self.last[2], st["tot_err"] - self.last[1]
                else: frate, dsym, derr = 0.0, 0, 0
                self.last, self.last_t = (st["frames"], st["tot_err"], st["tot_sym"]), now
                self.st = st
                if now - t_h >= 0.5 and self.base is not None:                      # the strips advance every 0.5 s
                    t_h = now
                    for h, v in ((hist_off, st["carrier_hz"]), (hist_fr, frate), (hist_ser, np.log10(max(derr / dsym, 1e-7)) if dsym > 0 else np.nan)): h.pop(0); h.append(v)
                self.win.append((now, st["tot_sym"], st["tot_err"])); self.win = [w for w in self.win if now - w[0] <= 2.0]
                ser_recent = float("nan")
                if len(self.win) > 1 and self.win[-1][1] > self.win[0][1]: ser_recent = (self.win[-1][2] - self.win[0][2]) / (self.win[-1][1] - self.win[0][1])
                b = self.base or {"tot_sym": 0, "tot_err": 0}
                ser_total = (st["tot_err"] - b["tot_err"]) / (st["tot_sym"] - b["tot_sym"]) if st["tot_sym"] > b["tot_sym"] else float("nan")
                self.image.value = render(self.z, self.hh, st, frate, ser_recent, ser_total, f"{rate:.1f} pictures/s, constellation of a frame {now - self.t_cap:.1f} s old")
                self.frames += 1; n_rate += 1
                if now - t_rate > 2: rate = n_rate / (now - t_rate); t_rate = now; n_rate = 0
            except Exception as e:
                self.status.value = f"{type(e).__name__}: {e}"; time.sleep(0.5)
        self.status.value = f"stopped after {self.frames} pictures"
    def start(self):
        if self.thread is None or not self.thread.is_alive():
            self.running = True; self.frames = 0
            self.thread = threading.Thread(target=self._run, daemon=True); self.thread.start()
            self.status.value = "running"
    def stop(self):
        self.running = False
        if self.thread is not None: self.thread.join(timeout=5)

### Picture self-test

A few pictures of the digital loopback: the display must say *DECODING* and show 16 clusters, and the EVM per subcarrier must be flat.

In [ ]:
p501_radio.loopback(ad, True); ad.set_tx_atten_db(TX_ATTEN0); tx_start(300); time.sleep(0.05)
rx.stop(); rx.configure(); rx.start(continuous=True, capture=True); rx.wait_frames(3); time.sleep(0.3)
img_w, st_w = widgets.Image(format="png"), widgets.Label()
lv = Live(img_w, st_w); lv.base = rx.stats()
lv.start(); time.sleep(3); lv.stop()
check("live pictures", lv.frames > 4 and len(img_w.value) > 2000, f"{lv.frames} pictures in 3 s ({lv.frames / 3.0:.1f}/s), status: {st_w.value}")
check("live: decoding, 16 clusters", lv.st is not None and lv.st["running"] and soft_evm_db(lv.z) < -30, f"soft EVM {soft_evm_db(lv.z):.1f} dB")
display(IPImage(img_w.value))
rx.stop(); tx_stop(); p501_radio.loopback(ad, False)

## Controls

**Receiver**
* **Start receiver** writes the detector settings, starts it (continuous: every frame the detector finds is decoded) and waits for the first frames. **Apply** writes the detector settings while it runs. **Stop receiver** stops it.
* **Detector threshold** (in 1/256 of the metric |S|^2/E^2: 166 = 0.65): lower finds weaker frames and gives more false detections (every false detection is thrown away by the check of the two training symbols and counted as *rejected*); higher needs a cleaner signal.
  **Target rms** is the level (counts) at which the samples enter the FFT after the gain, 400 by default; **Fixed gain** (Q8, 0 = from the detector) holds the gain whatever the detector measures (for experiments).
* **RX gain**: the AD9361's receive gain index. **Digital loopback**: the chip feeds the DAC into the receiver without an antenna (the transmit attenuation is then forced to its maximum: nothing is radiated).

**Transmitter** (the DAC DMA sends the stored frame as a cyclic buffer)
* **P501 transmits**: *off* or *OFDM frame*. **Level** = the rms of the DAC buffer in "ADC counts in the digital loopback" (65 .. 600; 500 over the air with **TX atten 20 to 30 dB**). **TX atten** is the transmit attenuation; the more, the weaker the signal at RX1.
  Changing the level restarts the DMA (a few frames are lost, and the cut-over frame can set the overflow flags `ovf` / `fft_ovf`: they stay set until the overlay is loaded again, i.e. until the *Configuration* cell is run again; the counters do not depend on them).

**Start display / Stop display**: the live picture; stopping it does not stop the receiver.

In [ ]:
w_rxg   = widgets.IntSlider(value=RX_GAIN, min=0, max=76, description="RX gain", continuous_update=False, layout=widgets.Layout(width="520px"))
w_loop  = widgets.Checkbox(value=False, description="Digital loopback")
w_tx    = widgets.Dropdown(options=[("off", 0), ("OFDM frame", 1)], value=0, description="P501 transmits", layout=widgets.Layout(width="260px"))
w_lvl   = widgets.IntSlider(value=300, min=65, max=600, step=5, description="Level", continuous_update=False, layout=widgets.Layout(width="520px"))
w_att   = widgets.FloatSlider(value=25.0, min=20.0, max=89.75, step=0.25, description="TX atten (dB)", continuous_update=False, layout=widgets.Layout(width="520px"))
w_thr   = widgets.IntSlider(value=166, min=100, max=230, step=2, description="Detector thr.", continuous_update=False, layout=widgets.Layout(width="520px"))
w_tgt   = widgets.IntSlider(value=400, min=100, max=800, step=10, description="Target rms", continuous_update=False, layout=widgets.Layout(width="520px"))
w_gov   = widgets.IntText(value=0, description="Fixed gain (Q8)", layout=widgets.Layout(width="260px"))
b_rx    = widgets.Button(description="Start receiver", button_style="success")
b_apply = widgets.Button(description="Apply", button_style="primary")
b_rxoff = widgets.Button(description="Stop receiver", button_style="warning")
b_start = widgets.Button(description="Start display", button_style="info")
b_stop  = widgets.Button(description="Stop display")
status  = widgets.Label(value="idle", layout=widgets.Layout(width="auto"))
errout  = widgets.Output(layout=widgets.Layout(border="1px solid #c0392b", max_height="200px", overflow="auto"))
image   = widgets.Image(format="png", width=900)
live    = Live(image, status)

def guard(fn):
    def h(change=None):
        try: fn(change)
        except Exception as e:
            status.value = f"{type(e).__name__}: {e}"
            with errout: print(traceback.format_exc())                          # the whole error, with the line, below the controls
    return h

def apply_live(_=None):
    """the detector controls into the registers (the receiver may be running)"""
    rx.configure(thr=w_thr.value, emin=4, target=w_tgt.value, gain_override=w_gov.value)
    status.value = f"set: detector threshold {w_thr.value}/256 = {w_thr.value / 256:.2f}, target rms {w_tgt.value}, fixed gain {w_gov.value if w_gov.value else 'from the detector'}"

def do_start_receiver(_=None):
    live.paused = True; time.sleep(0.15)                      # the live thread must not touch the registers meanwhile
    try:
        live.last = None
        for h in (hist_off, hist_fr, hist_ser): h[:] = [np.nan] * HIST
        rx.stop(); time.sleep(0.05); rx.configure(thr=w_thr.value, emin=4, target=w_tgt.value, gain_override=w_gov.value); rx.start(continuous=True, capture=True)
        t0 = time.time(); ok = rx.wait_frames(3, timeout_s=2.0); live.base = rx.stats()
        status.value = f"receiver started: {'frames found' if ok else 'no frame yet (is the transmitter on? is the LO the same as the B210 --freq?)'} after {time.time() - t0:.2f} s"
    finally:
        live.paused = False

def do_tx(change=None):
    ad.set_tx_atten_db(TX_ATTEN0 if w_loop.value else w_att.value)
    if w_tx.value == 1: tx_start(w_lvl.value)
    else: tx_stop(); p501_radio.quiet(ad, TX_ATTEN0)

def do_loop(change):
    p501_radio.loopback(ad, change["new"])
    if change["new"]: ad.set_tx_atten_db(TX_ATTEN0)               # nothing is radiated in the digital loopback
    elif w_tx.value == 1: ad.set_tx_atten_db(w_att.value)

b_rx.on_click(guard(do_start_receiver)); b_apply.on_click(guard(apply_live)); b_rxoff.on_click(guard(lambda b: rx.stop()))
b_start.on_click(guard(lambda b: live.start())); b_stop.on_click(guard(lambda b: live.stop()))
w_rxg.observe(guard(lambda c: ad.set_rx_manual_gain(c["new"])), names="value")
w_loop.observe(guard(do_loop), names="value")
w_tx.observe(guard(do_tx), names="value")
w_lvl.observe(guard(lambda c: tx_start(c["new"]) if w_tx.value else None), names="value")
w_att.observe(guard(lambda c: ad.set_tx_atten_db(c["new"]) if (w_tx.value and not w_loop.value) else None), names="value")

display(widgets.VBox([widgets.HBox([b_rx, b_apply, b_rxoff, b_start, b_stop, status]), w_thr, w_tgt, w_gov, w_rxg,
                      widgets.HBox([w_loop, w_tx]), w_lvl, w_att, errout, image]))

### Check the controls (optional)

Moves the controls from Python, the way clicks would, in the digital loopback: the registers must follow the controls and the receiver must find the frames.

In [ ]:
w_loop.value = True; w_lvl.value = 300; w_tx.value = 1; time.sleep(0.3)
w_thr.value = 150; w_tgt.value = 300; b_apply.click(); time.sleep(0.2)
check("apply writes the detector registers", rx.r("THR") == 150 and rx.r("TARGET") == 300 and rx.r("GAIN_OVR") == 0, f"THR {rx.r('THR')}, TARGET {rx.r('TARGET')}, GAIN_OVR {rx.r('GAIN_OVR')}")
w_thr.value = 166; w_tgt.value = 400
b_rx.click(); time.sleep(0.8); a = rx.stats(); time.sleep(1.0); b = rx.stats()
check("start receiver: frames found", b["frames"] - a["frames"] > 900 and b["running"], status.value)
check("decoding without an error", b["tot_err"] == a["tot_err"] and b["tot_sym"] - a["tot_sym"] > 900_000, f"{b['tot_err'] - a['tot_err']} errors in {b['tot_sym'] - a['tot_sym']:,} symbols, EVM {b['evm_db']:.1f} dB")
w_gov.value = 512; b_apply.click(); time.sleep(0.3); a = rx.stats(); time.sleep(0.5); b = rx.stats()
check("fixed gain 2.0 is used", abs(b["gain"] - 2.0) < 0.01 and b["frames"] - a["frames"] > 400, f"gain {b['gain']:.2f}, level {b['level_counts']:.0f} counts")
w_gov.value = 0; b_apply.click()
w_lvl.value = 100; time.sleep(0.5); a = rx.stats(); time.sleep(1.0); b = rx.stats()
check("level slider restarts the DMA at the new level", b["frames"] - a["frames"] > 900 and b["tot_err"] == a["tot_err"], f"{b['frames'] - a['frames']} frames, {b['tot_err'] - a['tot_err']} errors, level {b['level_counts']:.0f} counts (gain {b['gain']:.2f})")
b_rxoff.click(); w_tx.value = 0; w_loop.value = False
check("stop: receiver stopped, DMA stopped", not rx.stats()["running"] and _tx["buf"] is None, "")

## Try it: the B210 as the transmitter, and the air

1. Start the B210 transmitter on the host: `cd p501_fft_example && source tools/gnuradio_env.sh && python3 tutorial/ofdm/ofdm_frame_tx.py --freq 850e6 --tx-gain 79 --rms 0.25 --seconds 600` (antennas a few centimetres apart; keep the B210 gain at 79 dB or lower: it compresses at 85 dB). *LO* above must be the same frequency.
2. Untick *Digital loopback*, set *P501 transmits* to *off*, press **Start receiver**, then **Start display**. The B210 signal is found about 1000 times per second, the carrier offset is about **+9 kHz**, the EVM -19 to -23 dB (SNR 23 to 27 dB) and the symbol error rate between 0 and a few 1e-3 (see `tutorial/ofdm/results_board_air.txt`).
3. **Without the B210**: tick *Digital loopback* and set *P501 transmits* to *OFDM frame*: no error in millions of symbols and an EVM of -45 dB. Over the air from TX1 to RX1: untick the loopback, *Level* 500, *TX atten* 20 to 26 dB (the SNR falls with the attenuation: 19 dB at 22 dB attenuation).
4. Lower the **Detector threshold** or the **RX gain** and see where the frames are lost; look at the *EVM per data subcarrier* and *|H|* for the shape of the channel (the band edges at +-14 are weaker).

The next cell measures windows of 2 s like the board scripts (`p501_ofdm_b210.py`): it receives only and radiates nothing, so it is safe with the B210 running. The cell after it sweeps the TX attenuation over the air with the P501's own DAC (it radiates, so it only runs when `RUN_AIR = True`).

In [ ]:
RUN_B210 = False                    # True: 10 windows of 2 s of the signal that arrives now (start the B210 first)
def windows(n=10, seconds=2.0):
    live.stop(); rx.stop(); p501_radio.loopback(ad, False); ad.set_tx_atten_db(TX_ATTEN0) if w_tx.value == 0 else None
    rx.configure(); rx.start(continuous=True, capture=True); rx.wait_frames(3, timeout_s=2.0); time.sleep(0.2); a = rx.stats()
    print(f"{'window':>6}{'frames':>8}{'rejected':>9}{'bad':>6}{'sym errors':>12}{'SER':>10}{'EVM dB':>8}{'SNR dB':>8}{'carrier Hz':>11}{'level':>7}")
    for i in range(n):
        time.sleep(seconds); b = rx.stats(); d = lambda k: b[k] - a[k]
        print(f"{i:>6}{d('frames'):>8}{d('rejected'):>9}{d('bad_frames'):>6}{d('tot_err'):>12,}{d('tot_err') / max(d('tot_sym'), 1):>10.1e}{b['evm_db']:>8.1f}{b['snr_db']:>8.1f}{b['carrier_hz']:>11.0f}{b['level_counts']:>7.0f}")
        a = b
    rx.stop()
if RUN_B210: windows()
else: print("RUN_B210 is False: set it to True while the B210 transmits.")

In [ ]:
RUN_AIR = False                      # True: the P501 transmits over the air (TX1 -> RX1, antennas a few cm apart) at the attenuations below, weakest last
def air_experiment(attens=(20.0, 24.0, 28.0, 32.0), seconds=2.0, level=500):
    live.stop(); live.paused = True
    ad.set_rx_manual_gain(70); p501_radio.loopback(ad, False)
    print(f"{'TX atten':>9}{'frames':>8}{'rejected':>9}{'bad':>6}{'sym errors':>12}{'of symbols':>12}{'SER':>10}{'EVM dB':>8}{'SNR dB':>8}{'level':>7}")
    try:
        for att in attens:
            rx.stop(); ad.set_tx_atten_db(att); tx_start(level); time.sleep(0.1)
            rx.configure(); rx.start(continuous=True, capture=True); rx.wait_frames(3, timeout_s=2.0); time.sleep(0.3); a = rx.stats(); time.sleep(seconds); b = rx.stats(); d = lambda k: b[k] - a[k]
            print(f"{att:>7.0f}dB{d('frames'):>8}{d('rejected'):>9}{d('bad_frames'):>6}{d('tot_err'):>12,}{d('tot_sym'):>12,}{d('tot_err') / max(d('tot_sym'), 1):>10.1e}{b['evm_db']:>8.1f}{b['snr_db']:>8.1f}{b['level_counts']:>7.0f}")
    finally:
        rx.stop(); tx_stop(); p501_radio.quiet(ad, TX_ATTEN0); live.paused = False

if RUN_AIR: air_experiment()
else: print("RUN_AIR is False: nothing is transmitted. Set it to True to sweep the TX attenuation over the air.")

## Stop and silence

Run this when you are done (or after an error): stops the display and the receiver, the DMA and the digital loopback, and sets the maximum TX attenuation.
The AD9361 transmit chain stays powered (a weak LO carrier remains); loading another overlay or powering down gives complete silence.

In [ ]:
try:
    live.stop()
finally:
    rx.stop(); tx_stop()
    p501_radio.loopback(ad, False)
    p501_radio.quiet(ad, TX_ATTEN0)
    print("transmitter off, receiver stopped, loopback off, TX attenuation %.2f dB" % TX_ATTEN0)